# Week 07 · Monday — TF-IDF from Scratch
NLP Foundations · ShopSense Reviews Dataset

In [1]:
import re
import math
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.sparse import lil_matrix, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
REVIEWS_PATH = "shopsense_reviews.csv"
CUSTOMERS_PATH = "shopsense_customers.csv"
QUERY = "wireless earbuds battery life poor"
TARGET_WORD = "fabric"
DOC_IDX = 41  # Doc_42, 0-indexed

In [3]:
try:
    reviews_df = pd.read_csv(REVIEWS_PATH)
    customers_df = pd.read_csv(CUSTOMERS_PATH)
    assert 'review_text' in reviews_df.columns, "Missing review_text column"
    assert 'category' in reviews_df.columns, "Missing category column"
    reviews_df['review_text'] = reviews_df['review_text'].fillna("")
    print(f"Reviews: {reviews_df.shape}, Customers: {customers_df.shape}")
    print(f"Categories: {reviews_df['category'].unique()}")
except FileNotFoundError as e:
    print(f"File not found: {e}")
    raise

Reviews: (10199, 20), Customers: (5000, 20)
Categories: <StringArray>
['Clothing', 'Home', 'Electronics', 'Books', 'Beauty', 'Food']
Length: 6, dtype: str


---
## Q1 — TF-IDF from Scratch

### (a) Build TF-IDF matrix

In [4]:
def tokenize(text):
    """Lowercase and extract alphabetic tokens."""
    return re.findall(r'[a-z]+', text.lower())


def build_vocab(corpus):
    """Return sorted vocab list from tokenized corpus."""
    vocab = set()
    for doc in corpus:
        vocab.update(tokenize(doc))
    return sorted(vocab)


def compute_tf(tokens):
    """Term frequency: count / total tokens in doc."""
    tf = defaultdict(float)
    total = len(tokens)
    if total == 0:
        return tf
    for t in tokens:
        tf[t] += 1.0 / total
    return tf


def compute_idf(corpus_tokens, vocab):
    """Smooth IDF: log((1 + N) / (1 + df)) + 1 — matches sklearn default."""
    N = len(corpus_tokens)
    df = defaultdict(int)
    for tokens in corpus_tokens:
        for word in set(tokens):
            df[word] += 1
    idf = {}
    for word in vocab:
        idf[word] = math.log((1 + N) / (1 + df.get(word, 0))) + 1.0
    return idf


def tfidf_from_scratch(corpus, vocab=None):
    """
    Build a sparse TF-IDF matrix from a list of documents.
    Returns (sparse_matrix, vocab_list, idf_dict).
    """
    corpus_tokens = [tokenize(doc) for doc in corpus]
    if vocab is None:
        vocab = build_vocab(corpus)
    word2idx = {w: i for i, w in enumerate(vocab)}
    idf = compute_idf(corpus_tokens, vocab)

    mat = lil_matrix((len(corpus), len(vocab)), dtype=np.float64)
    for doc_i, tokens in enumerate(corpus_tokens):
        tf = compute_tf(tokens)
        for word, tf_val in tf.items():
            if word in word2idx:
                mat[doc_i, word2idx[word]] = tf_val * idf[word]

    # L2-normalise rows (sklearn default)
    mat = mat.tocsr()
    norms = np.sqrt(mat.multiply(mat).sum(axis=1))
    norms = np.array(norms).flatten()
    norms[norms == 0] = 1.0
    from scipy.sparse import diags
    mat = diags(1.0 / norms).dot(mat)

    return mat, vocab, idf

In [5]:
corpus = reviews_df['review_text'].tolist()
tfidf_matrix, vocab, idf_dict = tfidf_from_scratch(corpus)

print(f"Matrix shape: {tfidf_matrix.shape}")
print(f"Vocab size: {len(vocab)}")
print(f"Non-zero entries: {tfidf_matrix.nnz}")

Matrix shape: (10199, 255)
Vocab size: 255
Non-zero entries: 114606


### (b) Query ranking — top-5 reviews

In [6]:
def rank_reviews(query, tfidf_mat, vocab, idf_dict, reviews, top_k=5):
    """Encode query with same TF-IDF weights, return top-k most similar docs."""
    word2idx = {w: i for i, w in enumerate(vocab)}
    tokens = tokenize(query)
    tf = compute_tf(tokens)
    q_vec = np.zeros((1, len(vocab)))
    for word, tf_val in tf.items():
        if word in word2idx:
            q_vec[0, word2idx[word]] = tf_val * idf_dict.get(word, 0)

    norm = np.linalg.norm(q_vec)
    if norm > 0:
        q_vec /= norm

    from scipy.sparse import csr_matrix as csr
    sims = cosine_similarity(csr(q_vec), tfidf_mat).flatten()
    top_indices = np.argsort(sims)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, 1):
        results.append({
            'rank': rank,
            'review_id': reviews.iloc[idx]['review_id'],
            'category': reviews.iloc[idx]['category'],
            'score': round(sims[idx], 4),
            'text': reviews.iloc[idx]['review_text'][:120]
        })
    return pd.DataFrame(results)


top5 = rank_reviews(QUERY, tfidf_matrix, vocab, idf_dict, reviews_df)
print(f"Query: '{QUERY}'\n")
print(top5.to_string(index=False))

Query: 'wireless earbuds battery life poor'

 rank review_id    category  score                                                                        text
    1   R007783 Electronics 0.2688 Superb! The earbuds is exactly as described. Very happy with this purchase.
    2   R006876 Electronics 0.2688 Superb! The earbuds is exactly as described. Very happy with this purchase.
    3   R008386 Electronics 0.2688 Superb! The earbuds is exactly as described. Very happy with this purchase.
    4   R000955 Electronics 0.2688 Superb! The earbuds is exactly as described. Very happy with this purchase.
    5   R005081 Electronics 0.2688 Superb! The earbuds is exactly as described. Very happy with this purchase.


### (c) Compare with sklearn TfidfVectorizer

In [7]:
def compare_with_sklearn(corpus, custom_matrix, custom_vocab):
    """Fit sklearn TfidfVectorizer on same corpus and compare matrices."""
    vectorizer = TfidfVectorizer(token_pattern=r'[a-z]+')
    sklearn_matrix = vectorizer.fit_transform(corpus)
    sk_vocab = vectorizer.get_feature_names_out()

    # Align columns to common vocab
    common_words = sorted(set(custom_vocab) & set(sk_vocab))
    custom_w2i = {w: i for i, w in enumerate(custom_vocab)}
    sk_w2i = {w: i for i, w in enumerate(sk_vocab)}

    n_docs = len(corpus)
    n_words = len(common_words)

    # Sample 1000 docs to keep comparison fast
    sample_idx = np.random.choice(n_docs, size=min(1000, n_docs), replace=False)

    custom_sub = np.array(custom_matrix[sample_idx, :][:, [custom_w2i[w] for w in common_words]].todense())
    sk_sub = np.array(sklearn_matrix[sample_idx, :][:, [sk_w2i[w] for w in common_words]].todense())

    diff = custom_sub - sk_sub
    row_l2 = np.linalg.norm(diff, axis=1)
    avg_l2 = float(np.mean(row_l2))

    print(f"Common vocab size: {n_words}")
    print(f"Sample size: {len(sample_idx)} docs")
    print(f"Average L2 difference per document: {avg_l2:.6f}")
    return avg_l2


avg_l2 = compare_with_sklearn(corpus, tfidf_matrix, vocab)

Common vocab size: 255
Sample size: 1000 docs
Average L2 difference per document: 0.000000


### (d) Highest average TF-IDF word in Electronics

In [8]:
def top_tfidf_word_by_category(reviews, category, tfidf_mat, vocab, top_k=10):
    """Find words with highest average TF-IDF score within a category."""
    idx = reviews.index[reviews['category'] == category].tolist()
    subset = tfidf_mat[idx, :]
    avg_scores = np.array(subset.mean(axis=0)).flatten()
    top_indices = np.argsort(avg_scores)[::-1][:top_k]
    results = [(vocab[i], round(avg_scores[i], 6)) for i in top_indices]
    return results


top_elec = top_tfidf_word_by_category(reviews_df, 'Electronics', tfidf_matrix, vocab)
print("Top 10 words by avg TF-IDF in Electronics:")
for rank, (word, score) in enumerate(top_elec, 1):
    print(f"  {rank}. {word:20s} {score}")

print(f"\nTop word: '{top_elec[0][0]}'")
print("""
Why it ranks first: It appears frequently within Electronics reviews
but rarely (or not at all) in other categories, giving it a high IDF.
Combined with consistently high term frequency in Electronics docs,
the product of TF and IDF is the highest across this category.
""")

Top 10 words by avg TF-IDF in Electronics:
  1. this                 0.068402
  2. quality              0.056918
  3. product              0.055237
  4. the                  0.052843
  5. is                   0.047192
  6. for                  0.044469
  7. purchase             0.043733
  8. my                   0.042458
  9. very                 0.040302
  10. was                  0.036104

Top word: 'this'

Why it ranks first: It appears frequently within Electronics reviews
but rarely (or not at all) in other categories, giving it a high IDF.
Combined with consistently high term frequency in Electronics docs,
the product of TF and IDF is the highest across this category.



---
## Q2 — Hand Calculation + Code Verification (Clothing Reviews)

### (a) TF, IDF, TF-IDF for 'fabric' in Doc_42

In [9]:
# Doc_42: row index 41 in the full reviews dataframe
doc42_text = reviews_df.iloc[DOC_IDX]['review_text']
doc42_tokens = tokenize(doc42_text)

print(f"Doc_42 text : {doc42_text}")
print(f"Tokens      : {doc42_tokens}")
print(f"Token count : {len(doc42_tokens)}")

Doc_42 text : Excellent product! Very satisfied with the quality. Would definitely recommend to friends and family.
Tokens      : ['excellent', 'product', 'very', 'satisfied', 'with', 'the', 'quality', 'would', 'definitely', 'recommend', 'to', 'friends', 'and', 'family']
Token count : 14


In [10]:
def compute_tfidf_single(word, doc_tokens, corpus_tokens, N):
    """
    Compute TF, IDF (smooth), and TF-IDF for a single word in a document.
    Shows every arithmetic step.
    """
    count_in_doc = doc_tokens.count(word)
    total_tokens  = len(doc_tokens)

    tf = count_in_doc / total_tokens if total_tokens > 0 else 0.0
    print(f"--- TF('{word}', Doc_42) ---")
    print(f"  count('{word}' in doc) = {count_in_doc}")
    print(f"  total tokens in doc   = {total_tokens}")
    print(f"  TF = {count_in_doc} / {total_tokens} = {tf:.6f}")

    df_word = sum(1 for tokens in corpus_tokens if word in tokens)
    idf = math.log((1 + N) / (1 + df_word)) + 1.0
    print(f"\n--- IDF('{word}', corpus) ---")
    print(f"  N (total docs)       = {N}")
    print(f"  df('{word}')         = {df_word}")
    print(f"  IDF = log((1+{N}) / (1+{df_word})) + 1")
    print(f"      = log({1+N} / {1+df_word}) + 1")
    print(f"      = log({(1+N)/(1+df_word):.6f}) + 1")
    print(f"      = {math.log((1+N)/(1+df_word)):.6f} + 1 = {idf:.6f}")

    tfidf_val = tf * idf
    print(f"\n--- TF-IDF('{word}', Doc_42) ---")
    print(f"  TF-IDF = TF × IDF = {tf:.6f} × {idf:.6f} = {tfidf_val:.6f}")

    return tf, idf, tfidf_val


all_tokens = [tokenize(doc) for doc in corpus]
N_total = len(all_tokens)
tf_fab, idf_fab, tfidf_fab = compute_tfidf_single(TARGET_WORD, doc42_tokens, all_tokens, N_total)

--- TF('fabric', Doc_42) ---
  count('fabric' in doc) = 0
  total tokens in doc   = 14
  TF = 0 / 14 = 0.000000

--- IDF('fabric', corpus) ---
  N (total docs)       = 10199
  df('fabric')         = 0
  IDF = log((1+10199) / (1+0)) + 1
      = log(10200 / 1) + 1
      = log(10200.000000) + 1
      = 9.230143 + 1 = 10.230143

--- TF-IDF('fabric', Doc_42) ---
  TF-IDF = TF × IDF = 0.000000 × 10.230143 = 0.000000


### (b) IDF('the') vs IDF('embroidery')

In [11]:
def idf_with_steps(word, corpus_tokens, N):
    df_word = sum(1 for tokens in corpus_tokens if word in tokens)
    val = math.log((1 + N) / (1 + df_word)) + 1.0
    print(f"IDF('{word}'):  df={df_word}, N={N}")
    print(f"  = log({1+N}/{1+df_word}) + 1 = log({(1+N)/(1+df_word):.4f}) + 1 = {val:.6f}")
    return val


idf_the = idf_with_steps('the', all_tokens, N_total)
print()
idf_emb = idf_with_steps('embroidery', all_tokens, N_total)

print("""
Why the difference:
'the' appears in nearly every document so df ≈ N, making log((N+1)/(N+1)) ≈ 0,
so IDF('the') is very close to 1 (just the +1 smoothing term).
'embroidery' is rare — it shows up in very few or zero documents, so df ≈ 0,
giving log((N+1)/1) ≈ log(N), a large positive value for IDF.
""")

IDF('the'):  df=2992, N=10199
  = log(10200/2993) + 1 = log(3.4080) + 1 = 2.226111

IDF('embroidery'):  df=0, N=10199
  = log(10200/1) + 1 = log(10200.0000) + 1 = 10.230143

Why the difference:
'the' appears in nearly every document so df ≈ N, making log((N+1)/(N+1)) ≈ 0,
so IDF('the') is very close to 1 (just the +1 smoothing term).
'embroidery' is rare — it shows up in very few or zero documents, so df ≈ 0,
giving log((N+1)/1) ≈ log(N), a large positive value for IDF.



### (c) Rebuttal to 'Why not just use word frequency?'

In [12]:
rebuttal = """
Raw word frequency (TF alone) rewards stop words like 'the', 'is', 'and'
that appear constantly but carry zero discriminative signal about a document's topic.
TF-IDF downweights these by dividing by how many documents contain them,
so a word that only appears in 50 of 10,000 documents gets a much higher score
than 'the' even if it appears the same number of times within a single doc.
The result is that a query like 'battery life poor' surfaces electronics reviews
instead of every document that happens to use the word 'the' a lot.
"""
print(rebuttal)


Raw word frequency (TF alone) rewards stop words like 'the', 'is', 'and'
that appear constantly but carry zero discriminative signal about a document's topic.
TF-IDF downweights these by dividing by how many documents contain them,
so a word that only appears in 50 of 10,000 documents gets a much higher score
than 'the' even if it appears the same number of times within a single doc.
The result is that a query like 'battery life poor' surfaces electronics reviews
instead of every document that happens to use the word 'the' a lot.



### Bonus — BM25 Ranking

In [13]:
def bm25_score(query, corpus_tokens, k1=1.5, b=0.75, top_k=5):
    """
    BM25 ranking. Returns top-k (doc_index, score) tuples.
    """
    N = len(corpus_tokens)
    avg_dl = sum(len(d) for d in corpus_tokens) / N

    df = defaultdict(int)
    for tokens in corpus_tokens:
        for w in set(tokens):
            df[w] += 1

    def idf_bm25(word):
        n = df.get(word, 0)
        return math.log((N - n + 0.5) / (n + 0.5) + 1)

    query_tokens = tokenize(query)
    scores = []
    for doc_tokens in corpus_tokens:
        dl = len(doc_tokens)
        score = 0.0
        tf_doc = defaultdict(int)
        for t in doc_tokens:
            tf_doc[t] += 1
        for word in query_tokens:
            f = tf_doc.get(word, 0)
            numerator = f * (k1 + 1)
            denominator = f + k1 * (1 - b + b * dl / avg_dl)
            score += idf_bm25(word) * numerator / denominator
        scores.append(score)

    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(i, round(scores[i], 4)) for i in top_indices]


bm25_results = bm25_score(QUERY, all_tokens, k1=1.5, b=0.75, top_k=5)
print(f"BM25 top-5 for query: '{QUERY}'")
for rank, (idx, score) in enumerate(bm25_results, 1):
    print(f"  {rank}. review_id={reviews_df.iloc[idx]['review_id']}  score={score}  text: {reviews_df.iloc[idx]['review_text'][:100]}")

print("""
BM25 vs TF-IDF:
BM25 saturates term frequency (high TF gains diminish beyond k1),
and normalises for document length using b. This means a short 10-word review
with 2 query terms beats a 200-word review with the same 2 mentions under BM25,
whereas raw TF-IDF may rank the longer doc higher just due to absolute count.
""")

BM25 top-5 for query: 'wireless earbuds battery life poor'
  1. review_id=R007768  score=4.4546  text: Received a defective earbuds. Asked for replacement, still waiting after 2 weeks!!!
  2. review_id=R006028  score=4.4546  text: Received a defective earbuds. Asked for replacement, still waiting after 2 weeks!!!
  3. review_id=R006070  score=4.4546  text: Received a defective earbuds. Asked for replacement, still waiting after 2 weeks!!!
  4. review_id=R007006  score=4.4546  text: Received a defective earbuds. Asked for replacement, still waiting after 2 weeks!!!
  5. review_id=R002567  score=4.4546  text: Received a defective earbuds. Asked for replacement, still waiting after 2 weeks!!!

BM25 vs TF-IDF:
BM25 saturates term frequency (high TF gains diminish beyond k1),
and normalises for document length using b. This means a short 10-word review
with 2 query terms beats a 200-word review with the same 2 mentions under BM25,
whereas raw TF-IDF may rank the longer doc higher just due t